# Milestone 1: TuneWave Playlist
**Team Zyntra**

Here is our implementation for Milestone 1. We built a music playlist using a **Doubly Linked List (DLL)** paired with a **Hash Map (`dict`)**.

### Quick Recap of Rules:
1. Songs can only appear once in a playlist (we check this in $O(1)$ using the hash map).
2. With repeat **ON**, calling `next()` on the last song wraps back to the first song.
3. With repeat **OFF**, calling `next()` on the last song stays on the last song and returns `"end"`.
4. Required methods: `addSong(song)`, `removeSong(songId)`, `next()`, `previous()`, `toggleRepeat()`.


In [1]:
class Song:
    """Simple song model to hold song info."""
    def __init__(self, song_id, title, artist, duration=0):
        self.song_id = str(song_id)
        self.title = str(title)
        self.artist = str(artist)
        self.duration = duration

    def __repr__(self):
        return f"Song({self.song_id}, '{self.title}' - {self.artist})"

    def __eq__(self, other):
        if isinstance(other, Song):
            return self.song_id == other.song_id
        return self.song_id == str(other)

    def __hash__(self):
        return hash(self.song_id)


class SongNode:
    """Doubly linked list node for each song."""
    def __init__(self, song: Song):
        self.song = song
        self.prev = None
        self.next = None


class Playlist:
    """
    Playlist backed by Doubly Linked List + Hash Map (song_id -> node).
    All core operations run in O(1) time.
    """
    def __init__(self, name="My Playlist"):
        self.name = name
        self.head = None
        self.tail = None
        self.current = None
        self.repeat = False
        self.song_map = {}  # song_id -> SongNode for O(1) lookups
        self._count = 0

    def __len__(self):
        return self._count

    def addSong(self, song):
        """
        Adds a song to the end of playlist.
        Rejects duplicates in O(1) using the hash map.
        """
        # helper to accept Song object or dict or string id
        if not isinstance(song, Song):
            if isinstance(song, dict):
                song = Song(song.get("song_id") or song.get("id"), song.get("title", "Untitled"), song.get("artist", "Unknown"))
            elif isinstance(song, str):
                song = Song(song, song, "Unknown")
            else:
                return False

        # Rule 1: A song can appear only once in a playlist
        if song.song_id in self.song_map:
            # print(f"[Skip] Song '{song.title}' already exists in playlist.")
            return False

        node = SongNode(song)

        # edge case: first song in playlist
        if self.head is None:
            self.head = node
            self.tail = node
            self.current = node
        else:
            self.tail.next = node
            node.prev = self.tail
            self.tail = node

        self.song_map[song.song_id] = node
        self._count += 1
        return True

    def removeSong(self, songId):
        """
        Removes a song by ID in O(1) time without scanning.
        """
        sid = str(songId)
        node = self.song_map.get(sid)
        if not node:
            return False  # song not in playlist

        # if the deleted song is currently playing, move cursor
        if node == self.current:
            if node.next is not None:
                self.current = node.next
            elif node.prev is not None:
                self.current = node.prev
            else:
                self.current = None

        # unlink head / tail
        if node == self.head:
            self.head = node.next
            if self.head:
                self.head.prev = None

        if node == self.tail:
            self.tail = node.prev
            if self.tail:
                self.tail.next = None

        # unlink middle links
        if node.prev and node != self.head:
            node.prev.next = node.next
        if node.next and node != self.tail:
            node.next.prev = node.prev

        # cleanup
        del self.song_map[sid]
        self._count -= 1
        node.prev = None
        node.next = None
        return True

    def next(self):
        """
        Advances to next song.
        Rule: With repeat on, wraps to head. With repeat off, stays on last song and returns 'end'.
        """
        if self.current is None:
            return "end"

        if self.current.next is not None:
            self.current = self.current.next
            return self.current.song

        # At the last song
        if self.repeat:
            self.current = self.head  # wrap around
            return self.current.song if self.current else "end"
        else:
            # stays on the last song!
            return "end"

    def previous(self):
        """Moves back to the previous song in playlist."""
        if self.current is None:
            return None

        if self.current.prev is not None:
            self.current = self.current.prev
            return self.current.song

        # At the head
        if self.repeat:
            self.current = self.tail  # wrap to tail when repeat is on
            return self.current.song if self.current else None
        else:
            return self.current.song  # stay at head

    def toggleRepeat(self):
        """Toggles repeat mode on/off."""
        self.repeat = not self.repeat
        return self.repeat

    def getCurrentSong(self):
        return self.current.song if self.current else None


### Demo 1: Creating a Playlist and Testing Uniqueness
Testing adding songs and verifying duplicate prevention without scanning.


In [2]:
# Create a test playlist
my_playlist = Playlist("Night Vibes")

s1 = Song("S1", "Blinding Lights", "The Weeknd")
s2 = Song("S2", "Starboy", "The Weeknd")
s3 = Song("S3", "Midnight City", "M83")

print("Add s1:", my_playlist.addSong(s1))
print("Add s2:", my_playlist.addSong(s2))
print("Add s3:", my_playlist.addSong(s3))

# Try adding s1 again (duplicate)
print("Add s1 again (should be False):", my_playlist.addSong(s1))
print("Total songs in playlist:", len(my_playlist))


Add s1: True
Add s2: True
Add s3: True
Add s1 again (should be False): False
Total songs in playlist: 3


### Demo 2: Playback Navigation and Repeat Mode
Testing `next()` when repeat is OFF vs ON.


In [3]:
print("Currently playing:", my_playlist.getCurrentSong())

# Moving forward with repeat OFF
print("1st next():", my_playlist.next())
print("2nd next():", my_playlist.next())
print("3rd next() (at the end):", my_playlist.next())  # should return 'end'
print("Are we still on the last song?:", my_playlist.getCurrentSong())

# Moving back
print("\nNavigating backwards:")
print("previous():", my_playlist.previous())
print("previous():", my_playlist.previous())

# Now toggle repeat ON
print("\nToggling repeat mode:")
print("Repeat status:", my_playlist.toggleRepeat())

# Fast forward to end and check wrapping
my_playlist.next()
my_playlist.next()
print("At last song:", my_playlist.getCurrentSong())
print("Calling next() with repeat ON (should wrap to first song):", my_playlist.next())


Currently playing: Song(S1, 'Blinding Lights' - The Weeknd)
1st next(): Song(S2, 'Starboy' - The Weeknd)
2nd next(): Song(S3, 'Midnight City' - M83)
3rd next() (at the end): end
Are we still on the last song?: Song(S3, 'Midnight City' - M83)

Navigating backwards:
previous(): Song(S2, 'Starboy' - The Weeknd)
previous(): Song(S1, 'Blinding Lights' - The Weeknd)

Toggling repeat mode:
Repeat status: True
At last song: Song(S3, 'Midnight City' - M83)
Calling next() with repeat ON (should wrap to first song): Song(S1, 'Blinding Lights' - The Weeknd)


### Demo 3: Removing a Song While Playing
Testing removal of the active track in $O(1)$.


In [4]:
print("Active song before removal:", my_playlist.getCurrentSong())
# Remove currently active song
print("Removing active song S1:", my_playlist.removeSong("S1"))
print("Current song shifted to:", my_playlist.getCurrentSong())
print("Playlist length:", len(my_playlist))


Active song before removal: Song(S1, 'Blinding Lights' - The Weeknd)
Removing active song S1: True
Current song shifted to: Song(S2, 'Starboy' - The Weeknd)
Playlist length: 2


### Sanity Tests
Running checks on edge cases.


In [5]:
def run_tests():
    p = Playlist("Test")
    a = Song("A", "Song A", "Artist")
    b = Song("B", "Song B", "Artist")
    c = Song("C", "Song C", "Artist")

    # Add & duplicate
    assert p.addSong(a) is True
    assert p.addSong(b) is True
    assert p.addSong(a) is False  # duplicate rejected
    assert len(p) == 2

    # Traversal repeat OFF
    assert p.getCurrentSong() == a
    assert p.next() == b
    assert p.next() == "end"  # stays on last song
    assert p.getCurrentSong() == b

    # Previous
    assert p.previous() == a
    assert p.previous() == a  # stays at head

    # Repeat ON
    p.toggleRepeat()
    assert p.next() == b
    assert p.next() == a  # wrapped!

    # Remove
    assert p.removeSong("B") is True
    assert len(p) == 1
    assert p.removeSong("A") is True
    assert len(p) == 0
    assert p.getCurrentSong() is None
    assert p.next() == "end"

    print("All Milestone 1 tests passed successfully!")

run_tests()


All Milestone 1 tests passed successfully!
